# 09 — A controlled feature study

**Question.** Can broad feature engineering improve a compact ranker when the candidate pool,
historical cutoff, fitting sample, and evaluation cohort are held fixed?

**Result.** The selected 102-feature model achieves **0.58439 weighted Recall@20** on
**432,492 temporal evaluation sessions**. The compact 28-feature model scores **0.56490**;
fixed candidate fusion scores **0.53524**. Selection preferred removing source-score features.
Orders drive the gain; fusion still wins click and cart recall.

This notebook regenerates every chart from checksum-verified, committed evidence. Model fitting
and the independent event audit ran separately at full scale. [Notebook 10](10_competition_inference.ipynb)
runs native models and generates predictions. [Methods](../docs/RESEARCH.md) ·
[Machine-readable catalog](../reports/research/feature_catalog.csv) ·
[Independent audit](../reports/research/audit.json)

In [ ]:
from pathlib import Path
import hashlib
import json
import math
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'reports/research/manifest.json').is_file())
DATA = ROOT / 'reports/research'
load = lambda name: json.loads((DATA / name).read_text())
manifest = load('manifest.json')
for name, expected in manifest['files'].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == expected, name
evaluation, ablations = load('evaluation.json'), load('ablations.json')
audit, interpretation = load('audit.json'), load('interpretation.json')
screening, seal = load('screening.json'), load('evaluation_seal.json')
assert len({x['seal_id'] for x in (manifest, evaluation, audit, interpretation, seal)}) == 1
assert audit['status'] == evaluation['status'] == interpretation['status'] == 'passed'
assert seal['evaluation_labels_consulted'] is False
catalog = pd.read_csv(DATA / 'feature_catalog.csv')
families = pd.read_csv(DATA / 'feature_families.csv')
models = pd.read_csv(DATA / 'ablation_models.csv')
importance = pd.read_csv(DATA / 'feature_importance.csv')
COLORS = {'selected': '#0F766E', 'core': '#2563EB', 'fusion': '#64748B', 'ceiling': '#D97706'}
pio.renderers.default = 'plotly_mimetype+png'

def show(fig, title, height=480, left=90):
    fig.update_layout(template='plotly_white', width=1000, height=height,
        title=dict(text=title, x=0.03), font=dict(family='Arial', size=14, color='#1E293B'),
        margin=dict(l=left, r=40, t=85, b=70), paper_bgcolor='white', plot_bgcolor='white',
        legend=dict(orientation='h', y=1.12, x=0), hoverlabel=dict(font_size=14))
    fig.show()

print(f"Verified {len(manifest['files'])} evidence files; evaluation seal {seal['seal_id'][:12]}.")

## Freeze time before choosing models

Every learned retrieval component in this study is newly fitted on events before **20 August 2022,
22:00 UTC**. Whole sessions are assigned by their first event, clipped at exclusive period ends,
and cut into observed prefixes and future targets without using labels for inclusion. Fit and
selection session samples use deterministic hashes; evaluation includes every eligible session.

The original OTTO data had earlier exploratory exposure. This is a **newly reserved temporal
evaluation**, not a claim that the underlying data had never been seen. The earlier neural and
Item2Vec artifacts are not reused in this controlled comparison.

In [ ]:
protocol = load('temporal_contract.json')['protocol']
periods = pd.DataFrame([
    ['Historical retrieval', 'before 2022-08-20 22:00 UTC', 'all permitted history'],
    ['Ranker fit', '2022-08-20 → 2022-08-23, 22:00 UTC', f"{audit['source']['roles']['fit']:,} sessions"],
    ['Model selection', '2022-08-23 → 2022-08-24, 22:00 UTC', f"{audit['source']['roles']['selection']:,} sessions"],
    ['Final evaluation', '2022-08-24 → 2022-08-26, 22:00 UTC', f"{evaluation['sessions']:,} sessions"],
], columns=['Role', 'Time boundary', 'Size'])
display(periods.style.hide(axis='index'))
assert all(v == 0 for v in audit['source']['differences'].values())
assert audit['native_replay']['mismatches'] == 0
print(f"Independent audit: {audit['source']['source_partitions']} original Parquet partitions; "
      f"{audit['source']['reconstructed_labels']:,} reconstructed targets; "
      f"{audit['native_replay']['comparisons']:,} sampled native-model/candidate checks; zero differences.")

## Engineer broadly; retain evidence of value

The catalog spans historical popularity/trends, observed recurrence and intent, source agreement,
session context, and action/time-weighted graph affinity. Its 1,482 entries are explicit formulas,
not identifiers or future-label features. A broad catalog is a search space, not a result by itself.

Screening uses **514,013 candidate rows from 8,448 fitting sessions**. Constant, near-constant, and
duplicate columns are removed first. Nine binary LightGBM pilots—three objectives across three
session-grouped folds—supply normalized gain and stability diagnostics. Protected compact features,
redundancy filtering, and a 128-column capacity limit define the shortlist. Binary log loss and
feature/target correlations are screening diagnostics, not Recall@20 estimates.

In [ ]:
eligible = len(catalog) - sum(screening['rejections'][k] for k in ('constant', 'near_constant', 'duplicate'))
fig = go.Figure(go.Funnel(y=['Engineered formulas', 'Pass quality checks', 'Fit-only shortlist', 'Selection-chosen model'],
    x=[len(catalog), eligible, screening['retained_count'], int(catalog['final_selected'].sum())],
    textinfo='value+percent initial', marker=dict(color=['#CBD5E1', '#94A3B8', '#2563EB', '#0F766E'])))
show(fig, '1,482 engineered features → 102 in the final model', 430)
display(families.rename(columns={'family':'Family', 'engineered':'Engineered', 'screened':'Shortlist', 'final':'Final'})
        .style.hide(axis='index'))
rejected = pd.DataFrame([{'Reason': k, 'Features': v} for k, v in screening['rejections'].items() if k != 'retained'])
display(rejected.style.hide(axis='index'))
retained = catalog[catalog['status'] == 'retained']
display(retained.groupby('family', as_index=False).agg(
    median_fold_stability=('positive_gain_fold_fraction', 'median'),
    minimum_fold_stability=('positive_gain_fold_fraction', 'min')).style.format(precision=3).hide(axis='index'))

## Test the shortlist under a matched ranking experiment

Eight variants × three objectives = **24 LambdaRank fits**. Every variant uses the same
100,000 fitting sessions, 400-candidate pools, and retained positives plus the same 30 hard/30
random fitting negatives. All 20,000 selection queries keep the full candidate pool and complete
target denominators. Selection maximizes official task Recall@20; fewer features, then name,
break ties. Checkpoints are measured at iteration 1 and every 5 rounds, with early stopping.

Removing the 26 shortlisted source-score features wins all three objectives in selection.
Graph and intent interactions still encode retrieval information; this ablation isolates the
direct source family. This is a useful negative result: screening importance did not guarantee ranking utility.
The other five families remain, giving **102 features**. Family comparisons are exploratory
model-selection evidence; their maximum is not an unbiased performance estimate.

In [ ]:
selection = pd.DataFrame({name: {**{o: row['objectives'][o]['recall_at_20'] for o in ('clicks','carts','orders')},
    'weighted': row['weighted_recall_at_20']} for name,row in ablations['variants'].items()}).T
selection = selection.sort_values('weighted', ascending=False)
fig = px.imshow(selection, text_auto='.4f', aspect='auto', color_continuous_scale='Blues',
    labels=dict(x='Objective', y='Variant', color='Recall@20'))
fig.update_xaxes(side='top', title_text=None)
show(fig, 'Selection scores: same 20,000 sessions and 400 candidates', 570)
display(models[models['chosen']][['objective','variant','features','best_iteration','fit_seconds']]
    .sort_values('objective').style.format({'fit_seconds':'{:.2f}'}).hide(axis='index'))
assert set(ablations['chosen'].values()) == {'without_source'}
assert all(seal['models'][o]['variant'] == ablations['chosen'][o] for o in ablations['chosen'])
print('All model choices were sealed before evaluation labels were opened.')

## Evaluate once, with every eligible query in the denominator

The competition metric pools hits and unique-target denominators capped at 20 within each
objective, then applies weights **0.10 / 0.30 / 0.60**. Unretrievable targets remain in the
denominator. Click targets are the next click; carts and orders are unique future items.

The selected model improves the compact ranker by **1.949 percentage points** and fusion by
**4.915 points**. The paired 95% bootstrap intervals exclude zero. These intervals resample
sessions within this frozen model/cohort; they do not include training-seed uncertainty or
uncertainty from the search over model variants.

In [ ]:
scores = pd.DataFrame({name: {**{o: row['objectives'][o]['recall_at_20'] for o in ('clicks','carts','orders')},
    'weighted': row['weighted_recall_at_20']} for name,row in evaluation['scores'].items()})
for name,row in evaluation['scores'].items():
    recomputed = sum(w * row['objectives'][o]['hits'] / row['objectives'][o]['denominator']
        for o,w in {'clicks':.1,'carts':.3,'orders':.6}.items())
    assert math.isclose(recomputed, row['weighted_recall_at_20'], abs_tol=1e-12)
fig = go.Figure()
for name in ('fusion','core','selected'):
    fig.add_bar(name=name.capitalize(), x=scores.index, y=scores[name], marker_color=COLORS[name],
        text=scores[name].map(lambda x: f'{x:.4f}'), textposition='outside')
fig.update_layout(barmode='group')
fig.update_yaxes(title='Recall@20', range=[0,.78])
show(fig, f"Reserved temporal evaluation · {evaluation['sessions']:,} sessions", 510)
gain_rows = []
for baseline, result in evaluation['paired_intervals'].items():
    gain_rows.append({'Comparison': f'Selected − {baseline}', 'Gain (pp)': 100*result['absolute_gain'],
        '95% lower (pp)':100*result['gain_interval'][0], '95% upper (pp)':100*result['gain_interval'][1]})
display(pd.DataFrame(gain_rows).style.format(precision=3).hide(axis='index'))
print('1,000 paired session bootstrap replicates; seed 20260908.')

**What did not improve?** Fusion still beats the selected model on clicks (0.52678 vs 0.50565)
and carts (0.43021 vs 0.42792). Orders improve from 0.58917 to 0.67575 and carry 60% of the official
metric. No post-evaluation hybrid was selected to hide those losses. The old Fold 0 score in
Notebook 08 uses a different cohort and pool; it is not a matched baseline for this study.

## Separate candidate coverage from ranking quality

The same nested 100/200/400-item pools have weighted candidate ceilings of 0.65603 / 0.67497 /
0.68695. A ceiling counts recoverable targets with an ideal ordering, capped at 20; it is not an
achieved prediction score. All learned comparisons above use 400 candidates. These curves do not
claim that a ranker was refitted at each smaller budget or that larger pools have free compute cost.

In [ ]:
budgets = [100,200,400]
fig = go.Figure()
for objective in ('clicks','carts','orders'):
    y = [evaluation['candidate_frontier'][str(k)]['objectives'][objective]['recall_at_20'] for k in budgets]
    fig.add_scatter(name=objective.capitalize()+' ceiling', x=budgets, y=y, mode='lines+markers')
fig.add_scatter(name='Weighted ceiling', x=budgets,
    y=[evaluation['candidate_frontier'][str(k)]['weighted_recall_at_20'] for k in budgets],
    mode='lines+markers', line=dict(color=COLORS['ceiling'], width=4))
fig.add_scatter(name='Selected ranked score at 400', x=[400],
    y=[evaluation['scores']['selected']['weighted_recall_at_20']], mode='markers',
    marker=dict(color=COLORS['selected'], size=16, symbol='diamond'))
fig.update_xaxes(title='Candidates per query', tickvals=budgets)
fig.update_yaxes(title='Recall / candidate ceiling', range=[.48,.80])
show(fig, 'Candidate coverage places an upper bound on ranking quality', 510)

## Explain the frozen model without reopening selection

Native LightGBM TreeSHAP explains raw ranking scores on **4,096 deterministic selection
candidate rows**. Additivity is checked against native model outputs; the largest absolute error
is below 6 × 10⁻¹⁵. These contributions are not calibrated probabilities or causal effects.
Each panel uses its own objective's score scale.

In [ ]:
fig = make_subplots(rows=3, cols=1, subplot_titles=['Clicks','Carts','Orders'], vertical_spacing=.10)
for row, objective in enumerate(('clicks','carts','orders'), 1):
    top = importance[importance['objective'] == objective].nlargest(8, 'mean_absolute_shap').sort_values('mean_absolute_shap')
    fig.add_bar(x=top['mean_absolute_shap'], y=top['feature'], orientation='h', row=row, col=1,
        marker_color=COLORS['selected'], showlegend=False,
        hovertemplate='%{y}<br>Mean |SHAP|: %{x:.5f}<extra></extra>')
    fig.update_yaxes(tickfont=dict(size=12), row=row, col=1)
    fig.update_xaxes(title='Mean |SHAP|', row=row, col=1)
show(fig, 'Most influential features in each task-specific model', 1100, left=300)
display(pd.DataFrame([{'Objective': o, 'Maximum additivity error': e}
    for o,e in interpretation['shap_additivity_max_abs_error'].items()])
    .style.format({'Maximum additivity error':'{:.2e}'}).hide(axis='index'))

Whole-query block permutation complements feature-level SHAP. Each family is shuffled twice
across **1,000 selection sessions**, preserving relationships within the family. Candidate/feature
dependencies can still be broken, so the measured recall drop is a diagnostic. Two shuffles are
not a confidence interval. Context-only columns are constant within a query and can have little
direct ranking effect while interacting with item features.

In [ ]:
permutations = pd.DataFrame([{'family': family, 'mean': row['mean_weighted_recall_drop'],
    'minimum': min(row['repeat_drops']), 'maximum': max(row['repeat_drops'])}
    for family,row in interpretation['group_permutation'].items()]).sort_values('mean')
fig = go.Figure(go.Bar(x=permutations['mean']*100, y=permutations['family'], orientation='h',
    marker_color=COLORS['core'], error_x=dict(type='data', symmetric=False,
        array=(permutations['maximum']-permutations['mean'])*100,
        arrayminus=(permutations['mean']-permutations['minimum'])*100)))
fig.update_xaxes(title='Weighted Recall@20 drop (percentage points)')
show(fig, 'Repeat and intent evidence matter · whiskers span two shuffles', 440)
print(f"Diagnostic sample: {interpretation['selection_sessions']:,} sessions, "
      f"{interpretation['candidate_rows']:,} candidate rows; weighted recall {interpretation['sample_score']['weighted_recall_at_20']:.5f}.")

## Measure feature cost on identical work

The benchmark uses the same 32 selection prefixes and 400-candidate policy for every feature
set. The first pass warms the engine; two measured passes provide 64 query timings. It includes
candidate generation and requested feature construction in one process. It excludes model
prediction, storage/network overhead, and online serving. The hardware is a SageMaker
**ml.c7i.16xlarge (64 logical CPUs, approximately 124 GiB available RAM)**.

In [ ]:
timing = pd.DataFrame(interpretation['feature_benchmark']).T
labels = ['Broad catalog · 1,482','Shortlist · 128','Selected · 102']
fig = go.Figure()
for metric, color in [('p50_ms', COLORS['core']), ('p95_ms', COLORS['selected'])]:
    values = timing.loc[['broad_catalog','screened','selected_models'], metric]
    fig.add_bar(x=labels, y=values, name=metric.replace('_ms',''), marker_color=color,
        text=values.map(lambda x:f'{x:.2f} ms'), textposition='outside')
fig.update_layout(barmode='group')
fig.update_yaxes(title='Warm per-query feature time (ms)', range=[0,27])
show(fig, 'Computing selected features avoids most catalog-wide overhead', 440)
print(interpretation['benchmark_scope'])

## Inspect failures and set the limits of the claim

Short prefixes dominate the query count. The overall weighted metric pools task denominators;
it is not an unweighted average of the slice scores. In the 2–5-event slice the selected model
slightly loses to fusion, despite improving the overall result. Longer-session orders explain
much of the advantage. This is diagnostic analysis after the model was frozen, not another
model-selection pass.

In [ ]:
slices = []
for label in ('1','2-5','6-20','21+'):
    row = evaluation['prefix_slices'][label]
    slices.append({'Observed events':label, 'Sessions':row['sessions'],
        **{name:row[name]['weighted_recall_at_20'] for name in ('fusion','core','selected')}})
display(pd.DataFrame(slices).style.format({'Sessions':'{:,}', 'fusion':'{:.5f}', 'core':'{:.5f}', 'selected':'{:.5f}'}).hide(axis='index'))
checks = pd.DataFrame([
    ['Target reconstruction', f"{audit['source']['reconstructed_labels']:,} labels; zero differences"],
    ['Metric coverage', f"{audit['statistics']['parts']} parts; {audit['statistics']['sessions']:,} sessions"],
    ['Model selection', f"{audit['ablations']['native_models']} native models; predeclared rule verified"],
    ['Prediction replay', f"{audit['native_replay']['comparisons']:,} independent checks; zero differences"],
    ['Raw conversion provenance', 'Original raw JSONL identity retained; conversion not independently replayed'],
], columns=['Audit', 'Evidence'])
display(checks.style.hide(axis='index'))

## What this establishes

The experiment supports a measured gain from a selected 102-feature ranker over two matched
baselines, a meaningful negative source-family ablation, and a lower cost than materializing the
full catalog. Exact source hashes, checkpoint contracts, a pre-evaluation model seal, and an
independent event/metric audit make those claims traceable.

The study uses one fixed training seed, one newly reserved temporal cohort, and 100,000 fitting
sessions. It does not establish seed robustness, cross-period universality, causal feature effects,
online business lift, or state of the art. The underlying OTTO data had prior exploratory exposure.
Neural retrieval is separately documented in Notebooks 05–06; its earlier checkpoint is not part
of this certified-cutoff study. No Kaggle score or acceptance is claimed.

**Next in the review path:** [Notebook 10 — competition inference](10_competition_inference.ipynb)
replays the selected native models from committed example features and reproduces the full-data
generation workflow. [Model card](../docs/MODEL_CARD.md) · [Reproduce the experiment](../docs/REPRODUCIBILITY.md)